# Flow Procedure Graph – DDS Pipeline

This notebook generates a **flow procedure graph** summarizing the analysis pipeline
for TMS–EEG data (OpenNeuro ds001849) using the **Dual Damped Sine (DDS) model** 
and **cosine similarity**.

The purpose is to provide a **visual overview** of the methodological steps, including:

1. **Data ingestion**  
   - BIDS-structured EEG from OpenNeuro ds001849  
   - Active vs sham stimulation, sites: M1, DLPFC, PPC  

2. **Preprocessing**  
   - TMS artifact inpainting  
   - Filtering (notch, bandpass)  
   - Epoching and baseline correction  
   - ROI averaging  

3. **Modeling and metrics**  
   - DDS parameter estimation (A1, A2, f1, f2, γ1, γ2)  
   - Cosine similarity computation (Freedberg et al., 2020)  

4. **Statistical analysis**  
   - Mixed-effects or OLS + FE with cluster-robust SEs  
   - Condition × site effects  
   - Cross-method correlations  

5. **Visualization**  
   - Fit quality histograms and residuals  
   - Parameter distributions  
   - Correlation matrices  
   - Flow diagram (this notebook)  

---

### References

- **Damián Jan (2025)**. *A Dual Damped Sine (DDS) model for TMS–EEG: parametric validation against cosine similarity on OpenNeuro ds001849*. *Journal of Neuroscience Methods* (submitted).  
- **Freedberg et al., 2020**. *Identifying site- and stimulation-specific TMS-evoked EEG potentials using a quantitative cosine similarity metric*. *PLoS ONE*, 15(1): e0216185.  
  [https://doi.org/10.1371/journal.pone.0216185](https://doi.org/10.1371/journal.pone.0216185)

---

### Notes

- This notebook focuses only on the **workflow visualization**.  
- Graphs are generated in a **publication-ready format** (vector or high-res PNG).  
- Useful for inclusion in manuscripts, presentations, and project documentation.

---

📧 **Contact**: damian.jan@dejsl.com  
📝 **License**: MIT 


In [7]:
# pipeline_cosine_vs_dds_clean.py
# Matplotlib-only, journal-ready, improved layout (no overlaps).

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import textwrap
import pathlib

# --- journal defaults (monochrome) ---
mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.01,
    "font.family": "DejaVu Sans",  # o "Arial" si la tienes instalada
    "font.size": 9,
    "axes.linewidth": 0.8,
})

FIGSIZE = (7.6, 6)     # un poco más alto que antes
OUTNAME = "figure_pipeline_cosine_vs_dds"

def wrap(txt, width):
    return "\n".join(textwrap.wrap(txt, width=width, break_long_words=False, replace_whitespace=False))

def add_box(ax, x, y, w, h, text, lw=0.9, fc=0.97, rounded=True, fontsize=9, dash=False, wrap_width=None):
    if wrap_width:
        text = wrap(text, wrap_width)
    style = f"round,pad=0.02,rounding_size=0.03" if rounded else "square,pad=0.02"
    box = FancyBboxPatch((x, y), w, h, boxstyle=style,
                         linewidth=lw, edgecolor="black",
                         facecolor=str(fc) if isinstance(fc, float) else fc,
                         linestyle="--" if dash else "-",
                         mutation_aspect=1.0)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize)
    return box

def add_arrow(ax, x0, y0, x1, y1, lw=0.8, style="->", rad=0.0):
    arr = FancyArrowPatch((x0, y0), (x1, y1),
                          arrowstyle=style, mutation_scale=8,
                          linewidth=lw, color="black",
                          connectionstyle=f"arc3,rad={rad}")
    ax.add_patch(arr)
    return arr

def make_figure(outdir="."):
    outdir = pathlib.Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    fig = plt.figure(figsize=FIGSIZE)
    ax = fig.add_axes([0,0,1,1]); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")

    # --- Title (two lines) ---
    ax.text(0.5, 1.0, "Analysis pipeline: Cosine similarity vs DDS model",
            ha="center", va="top", fontsize=12)
    ax.text(0.5, 0.97, "(OpenNeuro ds001849, TMS–EEG)", ha="center", va="top", fontsize=10)

    # --- Common input / preprocessing (margen superior aumentado) ---
    b_input = add_box(ax, 0.30, 0.88, 0.40, 0.07,
                      "Preprocessed TMS–EEG epochs "
                      "(artifact handling, filtering, epoching, baseline)",
                      fc=0.96, wrap_width=48)

    b_avg = add_box(ax, 0.30, 0.79, 0.40, 0.07,
                    "Subject × site × condition averages (TEPs), 0–100 ms window",
                    fc=0.96, wrap_width=48)
    add_arrow(ax, 0.50, 0.88, 0.50, 0.79+0.01)

    # --- Header strip for branch labels (propio carril, sin solapar) ---
    ax.text(0.18, 0.730, "Cosine similarity pathway", ha="center", va="center", fontsize=9)
    ax.text(0.82, 0.745, "DDS pathway",               ha="center", va="center", fontsize=9)

    # --- Split arrows hacia cada rama ---
    add_arrow(ax, 0.50, 0.79, 0.22, 0.71, rad=0.0)
    add_arrow(ax, 0.50, 0.79, 0.78, 0.71, rad=0.0)

    # === Left branch: Cosine ===
    b_deriv = add_box(ax, 0.07, 0.65, 0.30, 0.07,
                      "Temporal derivative of TEP", fc=0.97, wrap_width=28)
    b_sign  = add_box(ax, 0.07, 0.56, 0.30, 0.07,
                      "Sign function / binarization (+1 / −1)",
                      fc=0.97, wrap_width=30)
    b_cos   = add_box(ax, 0.07, 0.46, 0.30, 0.08,
                      "Cosine similarity across sites "
                      "(M1–DLPFC, M1–PPC, DLPFC–PPC)",
                      fc=0.97, wrap_width=32)
    b_out_c = add_box(ax, 0.07, 0.35, 0.30, 0.07,
                      "Output: similarity index per subject × site × condition",
                      fc=0.93, dash=True, wrap_width=32)
    b_int_c = add_box(ax, 0.07, 0.23, 0.30, 0.09,
                      "Interpretation: global waveform overlap "
                      "(site-specificity proxy)",
                      fc=0.91, wrap_width=34)

    add_arrow(ax, 0.22, 0.71, 0.22, 0.65+0.01)
    add_arrow(ax, 0.22, 0.65, 0.22, 0.56+0.01)
    add_arrow(ax, 0.22, 0.56, 0.22, 0.46+0.01)
    add_arrow(ax, 0.22, 0.46, 0.22, 0.35+0.01)
    add_arrow(ax, 0.22, 0.35, 0.22, 0.23+0.01)

    # === Right branch: DDS ===
    b_dds   = add_box(ax, 0.63, 0.65, 0.30, 0.085,
                      "DDS: y(t)=A1·e^(−γ1 t)·sin(2π f1 t) + "
                      "A2·e^(−γ2 t)·sin(2π f2 t) + ε(t)",
                      fc=0.97, wrap_width=36)
    b_fit   = add_box(ax, 0.63, 0.55, 0.30, 0.075,
                      "Non-linear least squares fit "
                      "(goodness-of-fit: R², RMSE)",
                      fc=0.97, wrap_width=34)
    b_par   = add_box(ax, 0.63, 0.45, 0.30, 0.085,
                      "Parameters per subject × site × condition: "
                      "A1, A2, f1, f2, γ1, γ2",
                      fc=0.97, wrap_width=36)
    b_out_d = add_box(ax, 0.63, 0.35, 0.30, 0.07,
                      "Output: parameter table + fit metrics for statistics",
                      fc=0.93, dash=True, wrap_width=34)
    b_int_d = add_box(ax, 0.63, 0.23, 0.30, 0.09,
                      "Interpretation: mechanistic parameters of TEP dynamics "
                      "(amplitude, frequency, damping)",
                      fc=0.91, wrap_width=34)

    add_arrow(ax, 0.78, 0.71, 0.78, 0.65+0.01)
    add_arrow(ax, 0.78, 0.65, 0.78, 0.55+0.01)
    add_arrow(ax, 0.78, 0.55, 0.78, 0.45+0.01)
    add_arrow(ax, 0.78, 0.45, 0.78, 0.35+0.01)
    add_arrow(ax, 0.78, 0.35, 0.78, 0.23+0.01)

    # --- Complementarity banner ---
    ax.text(0.50, 0.14, "Complementary outputs → integrate for richer interpretation",
            ha="center", va="center", fontsize=9)

    # Save
    fig.savefig(outdir / f"{OUTNAME}.png")
    fig.savefig(outdir / f"{OUTNAME}.pdf")
    plt.close(fig)

if __name__ == "__main__":
    make_figure(outdir=".")
